# CT-to-ultrasound simulation with genuine Field II

This notebook downloads the public TotalSegmentator CT and segmentation reference pair when needed, runs the genuine Field II MATLAB backend, and visualizes the inputs and generated RF/B-mode outputs. It requires MATLAB plus the MATLAB Engine API for Python; it deliberately does not fall back to the Python reference backend.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'configs' / 'fieldii_matlab.yaml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'configs' / 'fieldii_matlab.yaml').exists():
    raise RuntimeError('Open this notebook from the ct2us_fieldii project or its notebooks directory.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'fieldii_matlab.yaml'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'fieldii_matlab'
PROJECT_ROOT

In [ ]:
from ct2us.dataset import download_reference_case

case_dir = PROJECT_ROOT / 'data' / 'totalsegmentator_example'
download_reference_case(case_dir)
sorted(path.name for path in case_dir.iterdir())

In [ ]:
from ct2us.pipeline import run

# This runs the same pipeline as: python -m ct2us.pipeline --config configs/fieldii_matlab.yaml
output_dir = run(CONFIG_PATH)
output_dir

In [ ]:
import json
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import yaml

with CONFIG_PATH.open() as stream:
    config = yaml.safe_load(stream)
ct = nib.load(case_dir / config['dataset']['ct_file']).get_fdata()
labels = nib.load(case_dir / config['dataset']['segmentation_file']).get_fdata().astype(int)
slice_index = ct.shape[2] // 2
ct_slice = ct[:, :, slice_index]
label_slice = labels[:, :, slice_index]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(ct_slice, cmap='gray', vmin=-1000, vmax=1000)
axes[0].set_title('Real CT slice (HU)')
axes[1].imshow(ct_slice, cmap='gray', vmin=-1000, vmax=1000)
axes[1].contour(label_slice > 0, levels=[0.5], colors='lime', linewidths=0.6)
axes[1].set_title('CT with anatomical-mask overlay')
for axis in axes:
    axis.axis('off')
plt.show()

In [ ]:
properties = np.load(OUTPUT_DIR / 'acoustic_properties_plane.npz')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, name, title in zip(axes, ['speed_m_s', 'impedance_rayl', 'diffuse_scatter'],
                             ['Speed of sound (m/s)', 'Impedance (Rayl)', 'Diffuse scattering']):
    image = axis.imshow(properties[name], cmap='viridis')
    axis.set_title(title)
    axis.axis('off')
    fig.colorbar(image, ax=axis, fraction=0.046)
plt.show()

In [ ]:
rf = np.load(OUTPUT_DIR / 'rf.npy')
bmode = np.load(OUTPUT_DIR / 'bmode.npy')
metadata = json.loads((OUTPUT_DIR / 'metadata.json').read_text())

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].imshow(rf, cmap='seismic', aspect='auto', vmin=-np.percentile(abs(rf), 99), vmax=np.percentile(abs(rf), 99))
axes[0].set_title('Simulated RF data')
axes[0].set_xlabel('Time sample')
axes[0].set_ylabel('Scan line')
axes[1].imshow(bmode.T, cmap='gray', origin='upper', aspect='auto',
               extent=[-metadata['bmode_lateral_extent_mm'] / 2, metadata['bmode_lateral_extent_mm'] / 2,
                       metadata['bmode_depth_mm'], 0])
axes[1].set_title('Envelope-detected, log-compressed B-mode')
axes[1].set_xlabel('Lateral position (mm)')
axes[1].set_ylabel('Depth (mm)')
plt.show()
metadata